# 🚀 Sổ Tay Qwen3-4B GPU T4 - Chuẩn Hóa Cho Colab

> **⚠️ BẮT BUỘC TRƯỚC KHI CHẠY:**
> 1. Vào menu **Runtime** &rarr; **Change runtime type** (Thay đổi loại phiên chạy)
> 2. Tại mục **Hardware accelerator**, chọn **T4 GPU** &rarr; Bấm **Save**.
> 3. Chọn **Runtime** &rarr; **Run all** (hoặc Ctrl + F9).
>
> *Mô hình đã được lưu trong Google Drive tại /content/drive/MyDrive/AI_Models/Qwen3-4B/Qwen3-4B-Q4_K_M.gguf, không cần tải lại.*

In [ ]:
# ==========================================================
# Ô 1: Kết nối Google Drive & Cài đặt thư viện hỗ trợ CUDA 12
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

print('⏳ Đang cài đặt thư viện llama-cpp-python (GPU cu124) và Gradio...')
# Cài đặt llama-cpp-python bản prebuilt CUDA 12.4
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Cài đặt các thư viện CUDA runtime và Gradio
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 gradio huggingface_hub
print('✅ [Ô 1 Xong] Đã kết nối Google Drive và cài đặt đầy đủ thư viện!')

In [ ]:
# ==========================================================
# Ô 2: Cấu hình & Nạp Driver GPU T4 (Sửa triệt để libcuda.so.1)
# ==========================================================
import os, sys, glob, site, ctypes, subprocess
from pathlib import Path

# 1. Kiểm tra phần cứng GPU NVIDIA
gpu_check = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if gpu_check.returncode != 0:
    print("❌ LỖI: Colab hiện ĐANG Ở CHẾ ĐỘ CPU, chưa bật GPU T4!")
    print("👉 Cách khắc phục:")
    print("   1. Nhìn lên menu trên cùng của Colab, chọn: Runtime -> Change runtime type")
    print("   2. Tại Hardware accelerator, chọn 'T4 GPU' -> Bấm 'Save'")
    print("   3. Sau đó chạy lại từ Ô 1!")
    raise RuntimeError("Vui lòng chuyển Runtime sang T4 GPU rồi chạy lại!")
else:
    print("✅ Đã kết nối GPU NVIDIA T4 thành công!")

# 2. Tìm và nạp libcuda.so.1 trực tiếp vào bộ nhớ toàn cục (RTLD_GLOBAL)
cuda_candidates = (
    glob.glob("/usr/local/cuda*/compat/libcuda.so*") +
    glob.glob("/usr/lib64-nvidia/libcuda.so*") +
    glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") +
    glob.glob("/usr/local/nvidia/lib64/libcuda.so*")
)

loaded_cuda = False
for cand in cuda_candidates:
    if os.path.isfile(cand):
        try:
            ctypes.CDLL(cand, mode=ctypes.RTLD_GLOBAL)
            print(f"✅ Đã nạp driver NVIDIA: {cand}")
            loaded_cuda = True
            break
        except Exception:
            pass

# 3. Tìm và nạp cuBLAS & cuDART từ site-packages
nvidia_dirs = set()
for base in site.getsitepackages():
    for f in Path(base).glob("nvidia/*/*/*.so*"):
        nvidia_dirs.add(str(f.parent))
        try:
            ctypes.CDLL(str(f), mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

# 4. Đăng ký vào ldconfig hệ thống Linux
all_paths = list(nvidia_dirs) + ["/usr/local/cuda/compat", "/usr/lib64-nvidia", "/usr/local/nvidia/lib64"]
existing_paths = [p for p in all_paths if os.path.exists(p)]
with open("/etc/ld.so.conf.d/nvidia_full.conf", "w") as conf:
    conf.write("
".join(existing_paths) + "
")
subprocess.run(["ldconfig"], check=True)

# 5. Kiểm tra nạp llama_cpp
import llama_cpp
print("🎉 [Ô 2 XONG] Thư viện llama-cpp-python đã kích hoạt thành công trên GPU T4!")


In [ ]:
# ==========================================================
# Ô 3: Nạp Qwen3-4B từ Google Drive vào GPU T4 & Chạy thử
# ==========================================================
import os
from pathlib import Path
from llama_cpp import Llama

# Đường dẫn file mô hình bạn đã lưu sẵn trong Drive
MODEL_PATH = '/content/drive/MyDrive/AI_Models/Qwen3-4B/Qwen3-4B-Q4_K_M.gguf'

# Kiểm tra file mô hình trong Google Drive
if not os.path.exists(MODEL_PATH):
    print(f'❌ Không tìm thấy model tại: {MODEL_PATH}')
    print('⏳ Tự động tải lại model từ Hugging Face vào Drive của bạn...')
    from huggingface_hub import hf_hub_download
    model_dir = Path('/content/drive/MyDrive/AI_Models/Qwen3-4B')
    model_dir.mkdir(parents=True, exist_ok=True)
    MODEL_PATH = hf_hub_download(
        repo_id='Qwen/Qwen3-4B-GGUF',
        filename='Qwen3-4B-Q4_K_M.gguf',
        local_dir=str(model_dir)
    )
    print('✅ Đã tải xong model vào Google Drive!')
else:
    print(f'✅ Đã tìm thấy model có sẵn trong Google Drive: {MODEL_PATH}')

print('⏳ Đang nạp Qwen3-4B vào GPU NVIDIA T4 (khoảng 15-20 giây)...')
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,      # Chạy 100% các lớp mô hình trên GPU T4
    n_ctx=2048,           # Độ dài context
    verbose=False
)
print('✅ [Ô 3 Xong] Nạp Qwen3-4B vào GPU thành công!')

# Thử nghiệm câu hỏi mẫu
print('⏳ Đang kiểm tra suy luận mẫu...')
test_resp = llm.create_chat_completion(
    messages=[
        {'role': 'system', 'content': 'Bạn là trợ lý AI hữu ích, trả lời bằng tiếng Việt.'},
        {'role': 'user', 'content': 'Xin chào, bạn là ai?'}
    ],
    max_tokens=100,
    temperature=0.7
)
print('👉 Phản hồi thử nghiệm của Qwen:
', test_resp['choices'][0]['message']['content'])

In [ ]:
# ==========================================================
# Ô 4: Khởi chạy API Gradio cho Laptop Fedora kết nối
# ==========================================================
import gradio as gr

def ask_qwen(message):
    result = llm.create_chat_completion(
        messages=[
            {'role': 'system', 'content': 'Bạn là trợ lý AI hữu ích, trả lời bằng tiếng Việt.'},
            {'role': 'user', 'content': message}
        ],
        max_tokens=512,
        temperature=0.7
    )
    return result['choices'][0]['message']['content']

# Cấu hình API endpoint tên là 'ask'
api = gr.Interface(
    fn=ask_qwen,
    inputs=gr.Textbox(label='Nhập câu hỏi'),
    outputs=gr.Textbox(label='Câu trả lời'),
    api_name='ask'
)

print('🚀 Đang mở đường hầm Gradio công khai...')
# Khởi chạy chia sẻ link công khai với tài khoản xác thực của bạn
api.launch(
    share=True,
    auth=('triet', 'QwenFedora7392!')
)